In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')


In [3]:
df.isnull().sum()


lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
numerical = [
        'annual_income',
        'number_of_courses_viewed',
        'interaction_count',
        'lead_score',
        ]

categorical = [
        'lead_source',
        'industry',
        'employment_status',
        'location'
        ]

In [10]:
df['industry'].value_counts()


industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
Name: count, dtype: int64

In [11]:
df['industry'].mode()

0    technology
Name: industry, dtype: str

In [12]:
df[numerical].corrwith(df.converted)


annual_income               0.181752
number_of_courses_viewed    0.395038
interaction_count           0.448624
lead_score                  0.490206
dtype: float64

In [13]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values


del df_train['converted']
del df_val['converted']
del df_test['converted']

In [15]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].fillna('NA').apply(mutual_info_converted_score)
mi = mi.sort_values(ascending=False)
mi

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

In [16]:
mi.index[0], round(mi.iloc[0], 2)

('lead_source', np.float64(0.03))

In [19]:
def get_filled(df):
    df_copy = df.copy()

    num_cols = [col for col in numerical if col in df_copy.columns]
    cat_cols = [col for col in categorical if col in df_copy.columns]

    df_copy[num_cols] = df_copy[num_cols].fillna(0.0)
    df_copy[cat_cols] = df_copy[cat_cols].fillna('NA')

    return df_copy

In [20]:
def train_model(df_1, df_2, y_1, y_2, c=1.0):
    dv = DictVectorizer(sparse=False)

    df_1 = get_filled(df_1)
    df_2 = get_filled(df_2)

    train_dict = df_1.to_dict(orient='records')
    val_dict = df_2.to_dict(orient='records')

    X_train = dv.fit_transform(train_dict)
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)

    model.fit(X_train, y_1)
    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_2, y_pred)

    return accuracy

In [21]:
result_4 = train_model(df_train, df_val, y_train, y_val)

In [23]:
round(result_4, 2)

0.65

In [24]:
fields = categorical + numerical
check_list = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count',
]

result_list = []

for d in check_list:
    #print(d)
    fields_copy = fields.copy()
    fields_copy.remove(d)

    df_train_copy = df_train.copy()
    df_val_copy = df_val.copy()

    df_1 = df_train_copy[fields_copy]
    df_2 = df_val_copy[fields_copy]

    result_5 = train_model(df_1, df_2, y_train, y_val)
    diff = result_4 - result_5

    result_list.append({"field": d, "diff": diff})

df_pred = pd.DataFrame(result_list)
min_result = df_pred.loc[df_pred['diff'].idxmin()]

In [25]:
min_result

field    number_of_courses_viewed
diff                        0.002
Name: 1, dtype: object

In [36]:
result_list_reg = []

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    result_5 = train_model(df_train, df_val, y_train, y_val, c=c)
    result_list_reg.append({"c": c, "accuracy": result_5})

df_pred_reg = pd.DataFrame(result_list_reg)
max_result = df_pred_reg.loc[df_pred_reg['accuracy'].idxmax()]

In [37]:
max_result

c           0.001
accuracy    0.645
Name: 3, dtype: float64